# State Objective Budget Gap Diagnostic V1

- **Baselines**: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `state_objective_budget_usage_bc_official_replay_v1`; reference `soxlbacktest_qlib_buysell_reference`.
- **Role**: diagnose.
- **Target window**: `wf_2020`, `wf_2022`, `wf_2023_2024`, `ytd_2026` through the dashboard replay safe end.
- **Changed layer**: model_selection / training_target.
- **Hypothesis**: the BC official replay gap is structural because the target under-specifies reentry participation after de-risking; running more PPO from this target should be blocked unless the gap is not structural.
- **Acceptance criteria**: identify whether the YTD return gap is dominated by low/zero exposure during major SOXL rebound windows and whether WF drawdown still violates the `<20%` retention gate.
- **Failure mode**: if the gap is structural, repeating PPO or adding another action cap keeps missing the same participation/drawdown joint objective.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_budget_gap import write_budget_usage_gap_diagnostic

SOURCE_CARD = ROOT / 'runtime/model_selection/card/state_objective_budget_usage_bc_official_replay_v1'
OUTPUT_DIR = ROOT / 'runtime/model_selection/card/state_objective_budget_gap_diagnostic_v1'
GATE_PATH = ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json'

gate_payload = json.loads(GATE_PATH.read_text())
gate = gate_payload['summary']
result = write_budget_usage_gap_diagnostic(
    source_card_dir=SOURCE_CARD,
    output_dir=OUTPUT_DIR,
    required_soxlbacktest_return_pct=float(gate['reference_return_pct']),
    required_best_return_pct=float(gate['required_best_return_pct']),
    max_walkforward_drawdown_pct=float(gate['max_walkforward_drawdown_pct']),
)
result.summary


{'experiment': 'state_objective_budget_gap_diagnostic_v1',
 'created_at': '2026-06-13T18:42:19.882922+00:00',
 'decision': 'revise_fold_local_reentry_participation_targets_before_ppo',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'ppo_allowed_from_current_target': False,
 'training_allowed_from_this_card': False,
 'source_experiment': 'state_objective_budget_usage_bc_official_replay_v1',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
 'required_soxlbacktest_return_pct': 132.077978,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'ytd_return_pct': 27.18605207710265,
 'ytd_gap_to_soxlbacktest_pp': 104.89192592289734,
 'ytd_gap_to_best_pp': 185.95740992289734,
 'largest_ytd_flat_positive_run_return_pct': 301.1137580990243,
 'structural_ytd_participation_gap': True,
 'worst_wf_max_drawdown_pct': 21.349091653065926,
 'walkforward_drawdown_gap': True,
 'next_mechanism': 'build_fold_local_reentry_participation_target_then_bc_

In [2]:
window_gap_summary = pd.read_csv(OUTPUT_DIR / 'window_gap_summary.csv')
window_gap_summary


,window,start,end,rows,return_pct,buyhold_return_pct,max_dd_pct,avg_position,flat_positive_run_count,largest_flat_positive_run_return_pct,largest_flat_positive_run_start,largest_flat_positive_run_end,largest_flat_positive_run_days
0,wf_2020,2020-01-02,2020-12-31,253,16.238860,355.736587,21.349092,0.240259,1,73.509405,2020-10-05,2020-12-31,62.0
1,wf_2022,2022-01-03,2022-12-30,251,-12.139386,-76.614267,14.800121,0.083689,0,NaN,NaN,NaN,NaN
2,wf_2023_2024,2023-01-03,2024-12-31,502,24.278118,55.654592,15.514461,0.165252,0,NaN,NaN,NaN,NaN
3,ytd_2026,2026-01-02,2026-06-09,109,27.186052,341.327496,17.820794,0.244897,1,301.113758,2026-03-12,2026-06-09,62.0


In [3]:
flat_positive_runs = pd.read_csv(OUTPUT_DIR / 'flat_positive_runs.csv')
flat_positive_runs.sort_values('return_pct', ascending=False).head(10)


,window,start,end,days,return_pct,start_price,end_price
1,ytd_2026,2026-03-12,2026-06-09,62,301.113758,50.279999,201.679993
0,wf_2020,2020-10-05,2020-12-31,62,73.509405,17.923332,31.098667


## Conclusion

- **Decision**: `revise_fold_local_reentry_participation_targets_before_ppo`.
- **Promotion / retention**: no actor promotion and no actor retention from this diagnostic.
- **Key evidence**: the official replay YTD return is `27.19%`, still `104.89pp` below SOXLBacktest and `185.96pp` below the current-best alignment floor. During `2026-03-12 -> 2026-06-09`, the replay stayed flat/near-flat while SOXL rose about `301.11%`.
- **Walk-forward risk**: `wf_2020` MaxDD is `21.35%`, which still violates the `<20%` retention gate by `1.35pp`; `wf_2022` and `wf_2023_2024` are below the drawdown cap but do not solve the YTD participation gap.
- **Next action**: build a fold-local reentry-participation target that explicitly separates de-risking from rebound re-entry before BC/PPO. Do not run another PPO from the current target as the next retained-model attempt.
- **Artifacts**: `summary.json`, `window_gap_summary.csv`, `flat_positive_runs.csv`.
